# Validação do artigo com Backlash-K0

Modelo multirrotor e integração usando `backlash_ross.py`, sem a adição da matriz linear `K_coupling`.

## 1. Importações

In [ ]:
import numpy as np
from backlash_ross import Backlash
from bifurcation_validation import build_multirotor, run_at_speed

## 2. Escolha da rotação e construção do multirrotor

In [ ]:
speed_rpm = 4500.0
multirotor = build_multirotor()
backlash, idx_x1 = run_at_speed(multirotor, speed_rpm, n_cicles=30, cut_cicles=0)
print(f'Modelo preparado para {speed_rpm:.0f} rpm')

## 3. Verificação da rigidez global K0

In [ ]:
model = backlash.multirotor
K_global = model.K(speed_rpm * np.pi / 30)
K_rotors = model._join_matrices(
    model.rotors['driving'].K(speed_rpm * np.pi / 30, speed_rpm * np.pi / 30),
    model.rotors['driven'].K(speed_rpm * np.pi / 30, speed_rpm * np.pi / 30 * model.mesh.gear_ratio),
)
np.testing.assert_allclose(K_global, K_rotors, rtol=0, atol=0)
print('K_global = K0: somente as rigidezes dos rotores estão presentes.')

## 4. Resultados do Newmark interno

In [ ]:
x1 = backlash.time_response.yout[:, idx_x1]
delta = backlash.backlash_results['delta']
force = backlash.backlash_results['Fm']
print(f'Amostras: {len(backlash.time)}')
print(f'max |x1| = {np.max(np.abs(x1)):.6e} m')
print(f'max |Fm| = {np.max(np.abs(force)):.6e} N')